# Canal de carros: teste de imagens
Mascote mecânico em estilo de brinquedo de blocos + carros esportivos japoneses (sem logos).
IP-Adapter plus-face para manter o mascote igual. Requer GPU T4 e Internet On.

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors

In [ ]:
import torch, os, shutil, time
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL = 'Lykon/dreamshaper-8'
OUT = '/kaggle/working/honda'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT, exist_ok=True)
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type='dpmsolver++', final_sigmas_type='sigma_min', use_karras_sigmas=True)
pipe = pipe.to('cuda')

In [ ]:
MASCOTE = 'solo, toy brick minifigure mechanic, yellow cylinder head, big smile, black slicked back hair, navy blue coveralls, oil stains'
ESTILO = 'toy brick style, glossy plastic, tilt shift miniature photo, soft studio light, vibrant colors, high detail'
NEGATIVO = ('two people, duplicate, crowd, logo, emblem, brand, text, watermark, letters, deformed, blurry, '
            'low quality, bad anatomy, extra limbs')

ref_prompt = f'{MASCOTE}, front view, upper body portrait, plain light grey background, {ESTILO}'
refs = []
for s in range(4):
    g = torch.Generator('cuda').manual_seed(200 + s)
    img = pipe(ref_prompt, negative_prompt=NEGATIVO, width=512, height=512,
               num_inference_steps=28, guidance_scale=7, generator=g).images[0]
    img.save(f'{OUT}/ref_{s}.png'); refs.append(img)
sheet = Image.new('RGB', (2048, 512), 'white')
for i, r in enumerate(refs): sheet.paste(r, (i * 512, 0))
sheet.save(f'{OUT}/refs_opcoes.png')
sheet.resize((1024, 256))

In [ ]:
REF_INDEX = 0
ref = refs[REF_INDEX]
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='models', weight_name='ip-adapter-plus-face_sd15.safetensors')

# (prompt, usa mascote?)
cenas = [
    (f'{MASCOTE}, medium shot, waving hello, standing in a cozy toy garage with tools on the wall', True),
    ('wide shot, small red 1970s japanese compact hatchback car made of toy bricks, parked on a toy city street', False),
    ('low angle shot, white 1990s japanese sports hatchback with rear spoiler made of toy bricks, racing on a toy race track, motion blur', False),
    (f'{MASCOTE}, close-up, holding a wrench, leaning on a red toy brick sports car, garage background', True),
    (f'{MASCOTE}, medium shot, thumbs up, next to a modern red japanese hatchback made of toy bricks, sunset road', True),
]
W, H = 768, 432
SEEDS = [42, 7]
grid = []
for i, (c, usa) in enumerate(cenas, 1):
    pipe.set_ip_adapter_scale(0.45 if usa else 0.0)
    row = []
    for v, sd in enumerate(SEEDS):
        g = torch.Generator('cuda').manual_seed(sd)
        img = pipe(f'{c}, {ESTILO}', negative_prompt=NEGATIVO, ip_adapter_image=ref,
                   width=W, height=H, num_inference_steps=28, guidance_scale=7.5, generator=g).images[0]
        img = img.resize((1536, 864), Image.LANCZOS)
        img.save(f'{OUT}/{i:03d}_{chr(97+v)}.png'); row.append(img)
    grid.append(row); print(f'cena {i:03d} ok')

In [ ]:
tw, th = 384, 216
contato = Image.new('RGB', (th + tw * 2, th * len(grid)), 'white')
for r, row in enumerate(grid):
    contato.paste(ref.resize((th, th)), (0, r * th))
    for v, im in enumerate(row):
        contato.paste(im.resize((tw, th)), (th + v * tw, r * th))
contato.save('/kaggle/working/folha_contato_honda.png')
shutil.make_archive('/kaggle/working/honda_cenas', 'zip', OUT)
contato